# Telco Dataset Integration Authoring

## 1. Authoring context and boundary declaration

This is the canonical Atlas dataset-integration authoring entrypoint for `telco-customer-churn`. It observes the exact current Atlas source independently, consumes S0165 structured scientific evidence only as authoring-time input, materializes the S0166 authoring suite, validates it with the generic validator, and prepares the S0167 capability-aware source/projection handoff.

It stops at the authoring boundary. Later pipeline stages own all modeling, candidate, publication, activation, and runtime responsibilities. The historical `01_dataset_authoring.ipynb` remains read-only provenance.

In [1]:
AUTHORING_BOUNDARY = {
    "atlas_source_observation_is_independent": True,
    "external_evidence_is_authoring_time_only": True,
    "external_analysis_is_not_reexecuted": True,
    "durable_absolute_external_root": False,
    "stops_before_downstream_release_and_runtime_stages": True,
}
assert AUTHORING_BOUNDARY["durable_absolute_external_root"] is False
assert all(value for key, value in AUTHORING_BOUNDARY.items() if key != "durable_absolute_external_root")

## 2. Dataset/source input identity
All durable inputs and outputs are repository-relative. The external root is an interactive session value and is never copied into an Atlas artifact.

In [2]:
from pathlib import Path
import hashlib
import json

repo_root = Path.cwd().resolve()
dataset_slug = "telco-customer-churn"
dataset_relative_path = "data/raw/telco-customer-churn.csv"
external_scientific_analysis_root = None  # session input only
external_evidence_index_relative_path = "evidence/external-analysis-evidence-index.json"
capability_profile_relative_path = "pipeline/capabilities/binary-predictive-classification.v1.json"
runtime_contract_relative_path = "contracts/telco-customer-churn/runtime-contract.json"
authoring_root_relative_path = "pipeline/authoring/telco-customer-churn"
authoring_generation_id = "telco-authoring-v1"
generated_at = "2026-08-07T00:00:00+00:00"

## 3. Atlas-owned source verification and drift checks
These observations describe what Atlas sees in the exact current CSV. They are not imported scientific conclusions.

In [3]:
from pipeline.discovery_evidence import (
    load_dataset_csv, observe_authoring_fields, resolve_repository_path,
    summarize_structure, summarize_target_column, summarize_identifier_columns,
)
dataset_path = resolve_repository_path(dataset_relative_path, repo_root=repo_root)
rows = load_dataset_csv(dataset_path)
atlas_structure = summarize_structure(rows)
assert atlas_structure["row_count"] == 7043
assert atlas_structure["column_count"] == 21
atlas_field_observations = observe_authoring_fields(rows, atlas_structure["ordered_columns"])
atlas_target_observation = summarize_target_column(rows, "Churn")
atlas_identifier_observation = summarize_identifier_columns(rows, ["customerID"])
assert set(atlas_target_observation["observed_labels"]) == {"No", "Yes"}
assert atlas_identifier_observation[0]["is_unique_per_row"]

FileNotFoundError: Dataset not found at: /home/fabyuu/Projetos/N8N/atlas-dataflow/notebooks/datasets/telco-customer-churn/data/raw/telco-customer-churn.csv

## 4. Structured external scientific-evidence discovery
The S0165 top-level index is located beneath an explicitly supplied external root. Atlas does not copy external notebooks or rerun that project.

In [ ]:
def safe_external_relative_path(value):
    candidate = Path(value)
    return bool(value) and not candidate.is_absolute() and ".." not in candidate.parts

external_root = Path(external_scientific_analysis_root).expanduser().resolve()
assert safe_external_relative_path(external_evidence_index_relative_path)
external_index_path = external_root / external_evidence_index_relative_path
external_evidence_index = json.loads(external_index_path.read_text(encoding="utf-8"))
assert external_evidence_index.get("artifact_type") in {"external_analysis_evidence_index", "atlas_compatible_external_analysis_evidence_index"}
assert external_evidence_index.get("dataset_identity", {}).get("dataset_slug") == dataset_slug

## 5. Evidence integrity/provenance verification
Every selected evidence reference is relative and hash-verified before semantic use. Only logical identity, relative identity, version/revision, and SHA-256 enter durable provenance.

In [ ]:
def sha256_file(path):
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for block in iter(lambda: handle.read(65536), b""):
            digest.update(block)
    return digest.hexdigest()

selected_external_evidence = []
for reference in external_evidence_index.get("evidence_references", []):
    relative_path = reference["relative_path"]
    assert safe_external_relative_path(relative_path)
    assert sha256_file(external_root / relative_path) == reference["sha256"]
    selected_external_evidence.append({
        "logical_producer_project_id": external_evidence_index["producer"]["logical_project_id"],
        "artifact_type": reference["artifact_type"],
        "artifact_version": reference["artifact_version"],
        "relative_path": relative_path,
        "sha256": reference["sha256"],
        "producer_revision_known": bool(external_evidence_index["producer"].get("revision")),
        "producer_revision": external_evidence_index["producer"].get("revision"),
    })

## 6. Dataset-specific semantic interpretation
Telco-specific field meaning, inclusion decisions, missing-value intent, categorical intent, target semantics, public meaning, and evidence rationale are materialized in `dataset-semantic-intent.v1`.

In [ ]:
feature_names = [name for name in atlas_structure["ordered_columns"] if name not in {"customerID", "Churn"}]
field_role_decisions = [{"field_name": "customerID", "role": "identifier", "include_in_features": False}]
field_role_decisions += [{
    "field_name": name, "role": "feature", "include_in_features": True,
    "missing_value_intent": ({"policy": "impute_fixed_value", "fixed_value": 0.0, "rationale": "Blank TotalCharges occurs with zero tenure."} if name == "TotalCharges" else {"policy": "no_missing_expected"}),
} for name in feature_names]
field_role_decisions.append({"field_name": "Churn", "role": "target", "include_in_features": False, "exclusion_reason": "Binary result field."})
semantic_intent = {
    "schema_version": "dataset-semantic-intent.v1", "artifact_type": "dataset_semantic_intent",
    "dataset_identity": {"dataset_slug": dataset_slug, "dataset_logical_name": "Telco Customer Churn"},
    "authoring_generation_id": authoring_generation_id,
    "governing_capability_profile": {"capability_profile_id": "binary-predictive-classification", "capability_profile_version": "v1"},
    "field_role_decisions": field_role_decisions,
    "target_semantics": {"target_field_name": "Churn", "task_type": "binary_classification", "positive_class": {"class_id": "Yes", "event_label": "customer churned"}, "is_final_training_configuration": False},
    "authored_public_meaning": {"human_reviewed": True, "safe_projection_intent": "Estimate customer churn propensity from reviewed service and account fields."},
    "authoring_rationale_refs": [{"reference_kind": "source_verification_evidence", "reference_id": "atlas-current-source-observation"}] + [{"reference_kind": "external_scientific_evidence", "reference_id": item["relative_path"]} for item in selected_external_evidence],
    "semantic_boundary_confirmations": {"observed_source_statistics_embedded": False, "scientific_conclusions_embedded": False, "training_outcome_embedded": False, "release_state_embedded": False, "model_bytes_embedded": False},
    "generated_at": generated_at,
}

## 7. Capability-profile selection/declaration
Selection is through the generic S0166 capability contract. No dataset-name branch and no new capability family is introduced.

In [ ]:
capability_profile_path = repo_root / capability_profile_relative_path
capability_profile = json.loads(capability_profile_path.read_text(encoding="utf-8"))
assert capability_profile["schema_version"] == "capability-profile.v1"
assert capability_profile["capability_profile_id"] == "binary-predictive-classification"
assert capability_profile["capability_profile_version"] == "v1"
assert capability_profile["support_status"] == "current_supported"

## 8. Deterministic preparation/input policy
The governed preparation role records the reviewed, deterministic TotalCharges rule and ordered source fields; it does not perform downstream modeling work.

In [ ]:
preparation_policy = {
    "schema_version": "candidate-preparation-recipe.v1",
    "dataset_slug": dataset_slug,
    "source_data_ref": dataset_relative_path,
    "ordered_input_columns": atlas_structure["ordered_columns"],
    "transformations": [{"field": "TotalCharges", "operation": "conditional_blank_to_zero", "when": {"field": "tenure", "equals": "0"}, "otherwise": "reject_blank"}],
    "deterministic": True,
}

## 9. S0166 authoring artifact materialization
The semantic intent and preparation policy are durable governed artifacts. The principal manifest coordinates them and Atlas source verification by safe relative references and hashes; external provenance contains no absolute root.

In [ ]:
def write_governed_json(relative_path, payload):
    path = repo_root / relative_path
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, sort_keys=True) + "\n", encoding="utf-8")
    return {"path": relative_path, "sha256": sha256_file(path)}

semantic_ref = write_governed_json(f"{authoring_root_relative_path}/dataset-semantic-intent.json", semantic_intent)
preparation_ref = write_governed_json(f"{authoring_root_relative_path}/preparation-policy.json", preparation_policy)
source_verification_ref = {"path": "pipeline/evidence/telco-customer-churn/discovery-evidence.json", "sha256": sha256_file(repo_root / "pipeline/evidence/telco-customer-churn/discovery-evidence.json")}
artifact_references = [
    {"role": "discovery_evidence", **source_verification_ref, "contract_version": "dataset-discovery-evidence.v1"},
    {"role": "semantic_intent", **semantic_ref, "contract_version": "dataset-semantic-intent.v1"},
    {"role": "preparation_policy", **preparation_ref, "contract_version": "candidate-preparation-recipe.v1"},
]
manifest = {
    "schema_version": "dataset-integration-authoring-manifest.v1", "artifact_type": "dataset_integration_authoring_manifest",
    "dataset_identity": {"dataset_slug": dataset_slug, "dataset_logical_name": "Telco Customer Churn"},
    "authoring_generation": {"authoring_generation_id": authoring_generation_id, "immutable": True, "generated_at": generated_at},
    "capability_profile_selection": {"capability_profile_id": capability_profile["capability_profile_id"], "capability_profile_version": capability_profile["capability_profile_version"], "capability_profile_ref": {"path": capability_profile_relative_path, "sha256": sha256_file(capability_profile_path)}},
    "artifact_references": artifact_references,
    "provenance": [{**item, "artifact_role": "external_scientific_evidence", "input_references": [], "generation_timestamp": None} for item in selected_external_evidence],
    "boundary_confirmations": {"complete_discovery_evidence_embedded": False, "complete_semantic_intent_embedded": False, "complete_preparation_recipe_embedded": False, "training_metrics_embedded": False, "model_selection_payload_embedded": False, "model_bytes_embedded": False, "inference_bundle_payload_embedded": False, "visual_payloads_embedded": False, "absolute_external_project_root_present": False, "external_analysis_handoff_replacement": False, "operational_importer_instruction_present": False},
    "generated_at": generated_at,
}
manifest_ref = write_governed_json(f"{authoring_root_relative_path}/dataset-integration-authoring-manifest.json", manifest)

## 10. Cross-artifact authoring validation
The generic S0166 validator checks schemas, identities, capability applicability, safe paths, and referenced hashes.

In [ ]:
from pipeline.authoring_contracts import validate_authoring_contracts
authoring_validation = validate_authoring_contracts(manifest, capability_profile, semantic_intent=semantic_intent, artifact_root=repo_root, expected_dataset_slug=dataset_slug, generated_at=generated_at)
assert authoring_validation.valid, authoring_validation.failures

## 11. S0167 capability-aware source/projection handoff
The source input carries the immutable authoring-generation and governed manifest/profile references. The generic S0167 projector resolves the capability boundary; the notebook does not implement capability-specific projection logic.

In [ ]:
from pipeline.contract_derivation import project_capability_aware_source_contract
source_contract_input = {
    "schema_version": "source-contract-input.v1", "dataset_slug": dataset_slug,
    "release_id": authoring_generation_id, "source_contract_ref": runtime_contract_relative_path, "source_data_ref": dataset_relative_path,
    "source_notebook_ref": "notebooks/datasets/telco-customer-churn/01_dataset_integration_authoring.ipynb",
    "authoring_generation_id": authoring_generation_id, "authoring_manifest_ref": {**manifest_ref, "contract_version": "dataset-integration-authoring-manifest.v1"},
    "capability_profile_id": capability_profile["capability_profile_id"], "capability_profile_version": capability_profile["capability_profile_version"],
    "capability_profile_ref": {"path": capability_profile_relative_path, "sha256": sha256_file(capability_profile_path)},
}
projection_handoff = project_capability_aware_source_contract(source_contract_input, repo_root=repo_root)
assert projection_handoff.authoring_boundary_valid is True

## 12. Authoring completion summary
Authoring is complete when source drift checks, external evidence integrity, S0166 cross-artifact validation, and the S0167 handoff boundary succeed. Continue downstream only through separately governed pipeline entrypoints.

In [ ]:
authoring_completion = {
    "dataset_slug": dataset_slug, "authoring_generation_id": authoring_generation_id,
    "source_verified_by_atlas": True, "external_evidence_integrity_verified": True,
    "authoring_contracts_valid": authoring_validation.valid,
    "capability_profile_id": capability_profile["capability_profile_id"],
    "next_boundary": "capability_aware_source_projection",
}
authoring_completion